# LocalAgentBench — Kaggle Setup & Preflight

**Goal:** prepare a CUDA-enabled llama.cpp, verify GPU/VRAM, and acquire + verify the pinned Qwen2.5-Coder-3B-Instruct checkpoint — WITHOUT running the experiment.

**Strict stop condition:** this notebook does not quantize and does not run the canary or the matrix. Quantization and the canary are separate, approved steps.

Expected runtime: a few minutes (plus the model download).

In [ ]:
# 0. Repo state
import os, subprocess, sys, hashlib, json, pathlib
ROOT = pathlib.Path('/kaggle/working')
REPO = ROOT / 'LocalAgentBench'
print('working dir:', ROOT)
print('exists repo:', REPO.exists())

## 1. Clone the pinned repository

Fill in PUBLIC_REPO_URL (public repo; no credentials). The notebook checks out the explicit commit hash recorded in the migration plan.

In [ ]:
PUBLIC_REPO_URL = os.environ.get('REPO_URL', '').strip()
PINNED_COMMIT = os.environ.get('REPO_COMMIT', '').strip()
assert PUBLIC_REPO_URL, 'Set REPO_URL environment variable to the public repo URL'
assert PINNED_COMMIT, 'Set REPO_COMMIT environment variable to the pinned commit SHA'
if not REPO.exists():
    subprocess.run(['git', 'clone', PUBLIC_REPO_URL], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', PINNED_COMMIT], cwd=REPO, check=True)
print('checked out', PINNED_COMMIT)
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO, check=True)

## 2. GPU + CUDA detection (dynamic)

Never assume a T4×2 or pooled memory. Record the actual GPU, VRAM, and CUDA version.

In [ ]:
import shutil
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'])
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
    print(out)
    lines = out.splitlines()
    vram_mib = [int(l.split(',')[1].strip().split()[0]) for l in lines]
    print('VRAM MiB per GPU:', vram_mib)
    # note: GPUs are NOT pooled; each process sees one GPU
else:
    print('ERROR: no nvidia-smi — not a GPU session'); sys.exit(1)

## 3. Build CUDA llama.cpp + preflight

Runs `scripts/setup_kaggle.sh` then `scripts/run_kaggle_preflight.sh`.

In [ ]:
subprocess.run(['bash', 'scripts/setup_kaggle.sh'], cwd=REPO, check=True)
subprocess.run(['bash', 'scripts/run_kaggle_preflight.sh'], cwd=REPO, check=True)

## 4. Model acquisition (restartable, pinned revision)

- Source: `Qwen/Qwen2.5-Coder-3B-Instruct` at revision `488639f1ff808d1d3d0ba301aef8c11461451ec5`.  
- License: Qwen Research (non-commercial). **Do not redistribute weights.**  
- We download only the **source checkpoint** (safetensors). Conversion to F16/quantized GGUF is a separate step.

We do **not** use third-party GGUF files.

In [ ]:
HF_REPO = 'Qwen/Qwen2.5-Coder-3B-Instruct'
HF_REV = '488639f1ff808d1d3d0ba301aef8c11461451ec5'
MODEL_DIR = REPO / 'models/source_qwen2_5_coder_3b_instruct'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
from huggingface_hub import snapshot_download  # available in Kaggle images
snapshot_download(
    repo_id=HF_REPO, revision=HF_REV, local_dir=str(MODEL_DIR),
    allow_patterns=['config.json','model-*.safetensors','model.safetensors.index.json',
                   'tokenizer*','merges.txt','vocab.json','generation_config.json','LICENSE'],
    max_workers=4,
)
print('downloaded to', MODEL_DIR)

## 5. Verify checkpoint integrity + write manifest

Compute sha256 of the downloaded safetensors and confirm the revision/license metadata before any conversion.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

st_files = sorted(MODEL_DIR.glob('model-*.safetensors'))
print('safetensors files:', [p.name for p in st_files])
manifest = {'repo': HF_REPO, 'revision': HF_REV,
           'license': 'qwen-research (non-commercial)',
           'files': {p.name: sha256(p) for p in st_files}}
manifest_path = REPO / 'manifests/model_provenance_kaggle.json'
manifest_path.write_text(json.dumps(manifest, indent=1))
print('manifest written:', manifest_path)
print(json.dumps({k: v for k,v in manifest.items() if k != 'files'}, indent=1))

## 6. Persist to Kaggle output

Copy repo state (configs, manifests, reports, scripts, downloaded model hashes) to `/kaggle/output` for durability, since Kaggle sessions can terminate.

In [ ]:
OUT = pathlib.Path('/kaggle/output')
OUT.mkdir(exist_ok=True)
shutil.copytree(REPO, OUT / 'LocalAgentBench', dirs_exist_ok=True)
print('copied to /kaggle/output')
# NOTE: model weights are NOT committed to git; they live under models/ (git-ignored).